<!-- # ARC-AGI Puzzle Solver Agent

This notebook implements an LLM-powered agent that plays the ARC-AGI puzzle game,
modeled after the TALE-Suite text adventure agent pattern.

The agent is given:
1. Training input-output grid pairs as examples
2. The test input grid
3. A restricted set of DSL functions (only those used in the ground-truth trajectory)

At each step the agent selects a DSL call to execute. The loop continues until
the produced grid matches the expected test output, or we reach the step limit.

## Setup

Required files in your working directory:
- `dsl.py`, `solvers.py`, `puzzle_ids.py`
- `data/` folder with `{puzzle_id}.json` files -->


In [55]:
import json, os, re, ast, time, copy, inspect, textwrap, requests
from typing import Dict, List, Any, Optional, Tuple


<!-- ## 1. Load the DSL and Constants -->


In [56]:
# ── Define constants used by solvers ──────────────────────────────
T, F = True, False
ZERO, ONE, TWO, THREE, FOUR, FIVE = 0, 1, 2, 3, 4, 5
SIX, SEVEN, EIGHT, NINE, TEN = 6, 7, 8, 9, 10
NEG_ONE, NEG_TWO = -1, -2
ORIGIN = (0, 0)
UNITY = (1, 1)
NEG_UNITY = (-1, -1)
DOWN, RIGHT, UP, LEFT = (1, 0), (0, 1), (-1, 0), (0, -1)
UP_RIGHT, DOWN_LEFT = (-1, 1), (1, -1)
TWO_BY_TWO = (2, 2)
THREE_BY_THREE = (3, 3)
TWO_BY_ZERO = (2, 0)
ZERO_BY_TWO = (0, 2)

CONSTANTS = {
    "T": T, "F": F,
    "ZERO": ZERO, "ONE": ONE, "TWO": TWO, "THREE": THREE,
    "FOUR": FOUR, "FIVE": FIVE, "SIX": SIX, "SEVEN": SEVEN,
    "EIGHT": EIGHT, "NINE": NINE, "TEN": TEN,
    "NEG_ONE": NEG_ONE, "NEG_TWO": NEG_TWO,
    "ORIGIN": ORIGIN, "UNITY": UNITY, "NEG_UNITY": NEG_UNITY,
    "DOWN": DOWN, "RIGHT": RIGHT, "UP": UP, "LEFT": LEFT,
    "UP_RIGHT": UP_RIGHT, "DOWN_LEFT": DOWN_LEFT,
    "TWO_BY_TWO": TWO_BY_TWO, "THREE_BY_THREE": THREE_BY_THREE,
    "TWO_BY_ZERO": TWO_BY_ZERO, "ZERO_BY_TWO": ZERO_BY_TWO,
}

# ── Write arc_types stub so dsl.py can import ────────────────────
arc_types_stub = (
    "from typing import Any, Callable, Container, FrozenSet, Tuple as Tuple\n"
    "Boolean = bool\n"
    "Integer = int\n"
    "Numerical = Any\n"
    "IntegerTuple = Tuple\n"
    "IntegerSet = Any\n"
    "ContainerContainer = Any\n"
    "Grid = Any\n"
    "Cell = Any\n"
    "Object = Any\n"
    "Objects = Any\n"
    "Indices = Any\n"
    "IndicesSet = Any\n"
    "Patch = Any\n"
    "Element = Any\n"
    "Piece = Any\n"
    "TupleTuple = Any\n"
)
with open("arc_types.py", "w") as f:
    f.write(arc_types_stub)

# ── Load DSL functions ────────────────────────────────────────────
DSL_SOURCE_PATH = "dsl.py"
SOLVERS_SOURCE_PATH = "solvers.py"

dsl_namespace: Dict[str, Any] = {}
dsl_namespace.update(CONSTANTS)
with open(DSL_SOURCE_PATH) as f:
    dsl_code = f.read()
exec(compile(dsl_code, DSL_SOURCE_PATH, "exec"), dsl_namespace)

DSL_FUNCTIONS: Dict[str, Any] = {
    k: v for k, v in dsl_namespace.items()
    if callable(v) and not k.startswith("_") and k[0].islower()
}

print(f"Loaded {len(DSL_FUNCTIONS)} DSL functions.")
print("Examples:", list(DSL_FUNCTIONS.keys())[:15])


Loaded 160 DSL functions.
Examples: ['identity', 'add', 'subtract', 'multiply', 'divide', 'invert', 'even', 'double', 'halve', 'flip', 'equality', 'contained', 'combine', 'intersection', 'difference']


<!-- ## 2. Extract DSL Signatures & Docstrings -->


In [57]:
def extract_dsl_docs(source_path: str) -> Dict[str, str]:
    """Extract function_name -> signature string from dsl.py."""
    with open(source_path) as f:
        source = f.read()
    pattern = r'def (\w+)\((.*?)\)\s*->\s*\w+:\s*"""(.*?)"""'
    matches = re.findall(pattern, source, re.DOTALL)
    docs = {}
    for name, raw_args, docstring in matches:
        arg_lines = [l.strip().rstrip(",") for l in raw_args.split("\n") if l.strip()]
        arg_names = [a.split(":")[0].strip() for a in arg_lines]
        sig = f"{name}({', '.join(arg_names)})"
        docs[name] = f"{sig}  --  {docstring.strip()}"
    return docs

DSL_DOCS = extract_dsl_docs(DSL_SOURCE_PATH)
print(f"Extracted docs for {len(DSL_DOCS)} functions.")
for k in list(DSL_DOCS)[:5]:
    print(f"  {DSL_DOCS[k]}")


Extracted docs for 160 functions.
  identity(x)  --  identity function
  add(a, b)  --  addition
  subtract(a, b)  --  subtraction
  multiply(a, b)  --  multiplication
  divide(a, b)  --  floor division


<!-- ## 3. Parse Solver Trajectories -->


In [58]:
def parse_solver(puzzle_id: str, source_path: str) -> dict:
    """Parse solve_{puzzle_id} to get DSL steps, function names, and constants."""
    with open(source_path) as f:
        source = f.read()

    pattern = rf'def solve_{puzzle_id}\(I\):\n(.*?)(?=\ndef |\Z)'
    m = re.search(pattern, source, re.DOTALL)
    if not m:
        raise ValueError(f"No solver found for {puzzle_id}")

    body = m.group(1)
    lines = [l.strip() for l in body.strip().split('\n')
             if l.strip() and not l.strip().startswith("return")]

    # AST-based extraction of function calls and constants
    func_source = f'def solve_{puzzle_id}(I):\n' + body
    tree = ast.parse(textwrap.dedent(func_source))
    dsl_funcs = set()
    constants_used = set()
    all_const_names = set(CONSTANTS.keys())

    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            dsl_funcs.add(node.func.id)
        if isinstance(node, ast.Name) and node.id in all_const_names:
            constants_used.add(node.id)

    return {
        "lines": lines,
        "dsl_funcs": dsl_funcs,
        "constants_used": constants_used,
        "num_steps": len(lines),
    }

# Quick test
info = parse_solver("00d62c1b", SOLVERS_SOURCE_PATH)
print("Puzzle 00d62c1b:")
print(f"  Steps: {info['num_steps']}")
print(f"  DSL funcs: {info['dsl_funcs']}")
print(f"  Constants: {info['constants_used']}")
print("  Lines:")
for l in info["lines"]:
    print(f"    {l}")


Puzzle 00d62c1b:
  Steps: 6
  DSL funcs: {'objects', 'rbind', 'colorfilter', 'compose', 'mfilter', 'fill'}
  Constants: {'F', 'ZERO', 'FOUR', 'T'}
  Lines:
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = rbind(bordering, I)
    x4 = compose(flip, x3)
    x5 = mfilter(x2, x4)
    O = fill(I, FOUR, x5)


<!-- ## 4. Puzzle Loader & Grid Utilities -->


In [59]:
def load_puzzle(puzzle_id: str, data_dir: str = "data") -> dict:
    """Load a puzzle JSON file."""
    path = os.path.join(data_dir, f"{puzzle_id}.json")
    with open(path) as f:
        return json.load(f)


def grid_to_tuple(grid_list):
    """Convert list-of-lists to tuple-of-tuples for DSL compatibility."""
    return tuple(tuple(row) for row in grid_list)


def grid_to_str(grid, indent=2) -> str:
    """Pretty-print a grid."""
    prefix = " " * indent
    return '\n'.join(prefix + str(list(row)) for row in grid)


def grids_match(a, b) -> bool:
    """Check if two grids are identical."""
    return tuple(tuple(r) for r in a) == tuple(tuple(r) for r in b)


def count_matching_cells(a, b) -> Tuple[int, int]:
    """Return (matching_cells, total_cells)."""
    a_t, b_t = [list(r) for r in a], [list(r) for r in b]
    if len(a_t) != len(b_t) or any(len(ar) != len(br) for ar, br in zip(a_t, b_t)):
        return (0, max(sum(len(r) for r in a_t), sum(len(r) for r in b_t)))
    total = sum(len(r) for r in a_t)
    match = sum(1 for i in range(len(a_t)) for j in range(len(a_t[i])) if a_t[i][j] == b_t[i][j])
    return (match, total)


# Quick test
puzzle = load_puzzle("00d62c1b")
print(f'Puzzle 00d62c1b: {len(puzzle["train"])} train pairs, {len(puzzle["test"])} test(s)')
print("Train pair 0 input:")
print(grid_to_str(puzzle["train"][0]["input"]))


Puzzle 00d62c1b: 5 train pairs, 1 test(s)
Train pair 0 input:
  [0, 0, 0, 0, 0, 0]
  [0, 0, 3, 0, 0, 0]
  [0, 3, 0, 3, 0, 0]
  [0, 0, 3, 0, 3, 0]
  [0, 0, 0, 3, 0, 0]
  [0, 0, 0, 0, 0, 0]


<!-- ## 5. Build the Initial Observation & System Prompt -->


In [ ]:
def build_system_prompt() -> str:
    """System prompt forcing strict ARC-DSL usage, no native Python/lambdas."""
    return (
        "You are an expert ARC-AGI puzzle solver. You are given a puzzle with "
        "training input-output pairs and a test input grid. Your goal is to "
        "produce the correct test output by applying DSL functions step by step.\n\n"
        "RULES:\n"
        "1. At each turn you see current variables and available DSL functions.\n"
        "2. Output EXACTLY ONE line of Python: `var = func(arg1, arg2, ...)`\n"
        "   Arguments must be existing variable names (I, x1, x2, ...) or constants.\n"
        "3. Name variables x1, x2, x3, ... Use O for the final output assignment.\n"
        "4. Study the training pairs to infer the pattern, then apply DSL functions.\n"
        "5. Output ONLY the code line — no explanation, no backticks, no comments.\n"
        "6. STRICT DSL COMPOSITION ONLY: You are strictly forbidden from writing native Python code. "
        "Do NOT write list comprehensions, do NOT use built-ins like `any`, `max`, or `sum`, "
        "and do NOT write `lambda` functions with mathematical logic. "
        "You must solve the puzzle entirely by composing the provided abstract ARC DSL primitives "
        "(e.g., `lbind`, `compose`, `matcher`, `adjacent`). Match the elegant style of a strict DSL solver.\n"
    )


def build_initial_observation(puzzle: dict, solver_info: dict) -> str:
    """Build the first observation showing training pairs, test input, and available DSLs."""
    parts = []
    parts.append("=== ARC-AGI PUZZLE ===")
    parts.append("TRAINING EXAMPLES:")
    for i, pair in enumerate(puzzle["train"]):
        parts.append(f"\n--- Train Pair {i+1} ---")
        parts.append(f'Input:\n{grid_to_str(pair["input"])}')
        parts.append(f'Output:\n{grid_to_str(pair["output"])}')

    parts.append("\n--- Test Input ---")
    parts.append(f'Input:\n{grid_to_str(puzzle["test"][0]["input"])}')
    parts.append("\nYour goal: produce the correct output grid for this test input.")

    parts.append("\n=== AVAILABLE DSL FUNCTIONS ===")
    for fname in solver_info.get("dsl_funcs_display", sorted(solver_info["dsl_funcs"])):
        if fname in DSL_DOCS:
            parts.append(f"  {DSL_DOCS[fname]}")
        else:
            parts.append(f"  {fname}(...)")

    if solver_info["constants_used"]:
        parts.append("\n=== AVAILABLE CONSTANTS ===")
        const_strs = [f'{c}={CONSTANTS[c]}' for c in sorted(solver_info['constants_used'])]
        parts.append("  " + ", ".join(const_strs))

    parts.append("\n=== CURRENT VARIABLES ===")
    parts.append("  I = <test input grid shown above>")
    parts.append("\nWrite the next DSL call:")
    return "\n".join(parts)


# Quick test
solver_info = parse_solver("00d62c1b", SOLVERS_SOURCE_PATH)
puzzle = load_puzzle("00d62c1b")
obs = build_initial_observation(puzzle, solver_info)
print(obs[:5000])
print("...")


=== ARC-AGI PUZZLE ===
TRAINING EXAMPLES:

--- Train Pair 1 ---
Input:
  [0, 0, 0, 0, 0, 0]
  [0, 0, 3, 0, 0, 0]
  [0, 3, 0, 3, 0, 0]
  [0, 0, 3, 0, 3, 0]
  [0, 0, 0, 3, 0, 0]
  [0, 0, 0, 0, 0, 0]
Output:
  [0, 0, 0, 0, 0, 0]
  [0, 0, 3, 0, 0, 0]
  [0, 3, 4, 3, 0, 0]
  [0, 0, 3, 4, 3, 0]
  [0, 0, 0, 3, 0, 0]
  [0, 0, 0, 0, 0, 0]

--- Train Pair 2 ---
Input:
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  [0, 0, 3, 0, 3, 0, 0, 0, 0, 0]
  [0, 0, 0, 3, 0, 3, 0, 0, 0, 0]
  [0, 0, 3, 0, 0, 0, 3, 0, 0, 0]
  [0, 0, 0, 0, 0, 3, 0, 3, 0, 0]
  [0, 0, 0, 3, 0, 3, 3, 0, 0, 0]
  [0, 0, 3, 3, 3, 0, 0, 0, 0, 0]
  [0, 0, 0, 3, 0, 0, 0, 0, 0, 0]
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Output:
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  [0, 0, 3, 0, 3, 0, 0, 0, 0, 0]
  [0, 0, 0, 3, 0, 3, 0, 0, 0, 0]
  [0, 0, 3, 0, 0, 0, 3, 0, 0, 0]
  [0, 0, 0, 0, 0, 3, 4, 3, 0, 0]
  [0, 0, 0, 3, 0, 3, 3, 0, 0, 0]
  [0, 0, 3, 3, 3, 0, 0, 0, 0, 0]
  [0, 0, 0, 3, 0, 0, 0, 0, 0, 0]
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  [0,

<!-- ## 6. DSL Execution Engine -->


In [61]:
class DSLExecutor:
    """Maintains variable state and executes DSL calls step by step."""

    def __init__(self, test_input_grid, solver_info: dict):
        self.variables: Dict[str, Any] = {
            "I": grid_to_tuple(test_input_grid),
        }
        self.solver_info = solver_info
        self.exec_ns: Dict[str, Any] = {}
        self.exec_ns.update(CONSTANTS)
        self.exec_ns.update(DSL_FUNCTIONS)

    def execute_line(self, line: str) -> Tuple[bool, str]:
        """Execute a single DSL assignment. Returns (success, message)."""
        line = line.strip().strip("`").strip()
        if line.startswith("python"):
            line = line[6:].strip()

        if "=" not in line:
            return False, f'Invalid format (no assignment): {line}'

        match = re.match(r'^(\w+)\s*=\s*(.+)$', line)
        if not match:
            return False, f'Could not parse: {line}'

        var_name = match.group(1)
        # Allow a safe subset of builtins so the LLM can use basic Python
        # operations (list concat, type conversions, etc.) alongside DSL calls
        safe_builtins = {
            'list': list, 'tuple': tuple, 'set': set, 'frozenset': frozenset,
            'int': int, 'float': float, 'str': str, 'bool': bool,
            'len': len, 'range': range, 'enumerate': enumerate, 'zip': zip,
            'min': min, 'max': max, 'sum': sum, 'sorted': sorted,
            'abs': abs, 'round': round,
            'True': True, 'False': False, 'None': None,
            'isinstance': isinstance, 'type': type,
            'map': map, 'filter': filter,
        }
        local_ns = {}
        local_ns.update(self.exec_ns)
        local_ns.update(self.variables)

        def _try_exec(line, builtins_dict, ns):
            ns['__builtins__'] = builtins_dict
            exec(line, ns)
            if var_name in ns:
                return ns[var_name]
            return None


        try:
            result = _try_exec(line, safe_builtins, local_ns)
        except TypeError:
            # Retry: wrap DSL funcs so frozenset returns become tuples,
            # which fixes list + frozenset concatenation issues.
            def _wrap(fn):
                def wrapper(*a, **kw):
                    r = fn(*a, **kw)
                    return tuple(r) if isinstance(r, frozenset) else r
                return wrapper
            retry_ns = dict(local_ns)
            for k in list(retry_ns):
                if callable(retry_ns[k]) and k in DSL_FUNCTIONS:
                    retry_ns[k] = _wrap(retry_ns[k])
            # Also convert any frozenset variables to tuples
            for k in list(retry_ns):
                if isinstance(retry_ns[k], frozenset):
                    retry_ns[k] = tuple(retry_ns[k])
            try:
                result = _try_exec(line, safe_builtins, retry_ns)
            except Exception as e2:
                return False, f'Execution error: {type(e2).__name__}: {e2}'
        except Exception as e:
            return False, f'Execution error: {type(e).__name__}: {e}'

        if result is not None:
            # If the LLM returned a raw grid (list-of-lists), convert to
            # tuple-of-tuples for consistency with DSL internal format.
            if (isinstance(result, (list, tuple)) and len(result) > 0
                    and isinstance(result[0], (list, tuple))):
                result = tuple(tuple(row) for row in result)
            self.variables[var_name] = result
            return True, f'OK: {var_name} assigned'
        else:
            return False, f'Variable {var_name} not found after execution'

    def get_state_summary(self) -> str:
        """Return a text summary of current variables for the LLM."""
        parts = ["=== CURRENT VARIABLES ==="]
        for name, val in self.variables.items():
            if isinstance(val, tuple) and len(val) > 0 and isinstance(val[0], tuple):
                rows, cols = len(val), len(val[0]) if val else 0
                parts.append(f'  {name} = <{rows}x{cols} grid>')
                parts.append(grid_to_str(val, indent=4))
            else:
                val_str = str(val)
                if len(val_str) > 200:
                    val_str = val_str[:200] + "..."
                parts.append(f'  {name} = {val_str}')
        return "\n".join(parts)

    def has_output(self) -> bool:
        return "O" in self.variables

    def get_output(self):
        return self.variables.get("O", None)


# Quick test
executor = DSLExecutor(puzzle["test"][0]["input"], solver_info)
ok, msg = executor.execute_line("x1 = objects(I, T, F, F)")
print(f'Step 1: {ok}, {msg}')
ok, msg = executor.execute_line("x2 = colorfilter(x1, ZERO)")
print(f'Step 2: {ok}, {msg}')
print(executor.get_state_summary()[:500])


Step 1: True, OK: x1 assigned
Step 2: True, OK: x2 assigned
=== CURRENT VARIABLES ===
  I = <20x20 grid>
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    [0, 3, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    [0, 0, 3, 0, 3, 3, 3, 3, 3, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0]
    [0, 0, 0, 0, 3, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0]
    [0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0]
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0]



<!-- ## 7. ARC Agent (LLM-powered)

The agent mimics the TALE-Suite `APIReasoningAgent` pattern:
system prompt + conversation history + API calls with retry. -->


In [62]:
class ARCAgent:
    """LLM agent that plays the ARC-AGI game via iterative DSL calls."""

    def __init__(self, api_url: str, api_key: str, model: str,
                 seed: int = 42, temperature: float = 0.0, max_tokens: int = 10240):
        self.api_url = api_url
        self.api_key = api_key
        self.model = model
        self.seed = seed
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.history: List[dict] = []

    def _api_call(self, messages: List[dict]) -> dict:
        """Chat completion request to the Triton API."""
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }
        payload = {
            "model": self.model,
            "messages": messages,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
            "max_completion_tokens": self.max_tokens,
            "seed": self.seed,
        }
        response = requests.post(self.api_url, headers=headers, json=payload, timeout=300)
        return response.json()

    def _call_with_retry(self, messages, max_retries=5, base_sleep=1.0):
        """Call API with exponential backoff retry."""
        last_response = None
        for attempt in range(max_retries):
            try:
                last_response = self._api_call(messages)
                if "choices" in last_response and "usage" in last_response:
                    return last_response
            except Exception as e:
                print(f'  API attempt {attempt+1} failed: {e}')
            if attempt < max_retries - 1:
                time.sleep(base_sleep * (2 ** attempt))
        return last_response

    def reset(self, system_prompt: str):
        """Reset agent for a new puzzle."""
        self.history = [{"role": "system", "content": system_prompt}]

    def act(self, observation: str) -> Tuple[str, dict]:
        """Given an observation, query the LLM and return (action_line, stats)."""
        self.history.append({"role": "user", "content": observation})

        print(f"Current conversation history (last 2 messages):")
        for msg in self.history[-2:]:
            print(f"  {msg['role']}: {msg['content'][:200]}{'...' if len(msg['content']) > 200 else ''}")

        response = self._call_with_retry(self.history)

        if response is None:
            action = "O = I"
            stats = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
        else:
            content = (response.get("choices", [{}])[0]
                       .get("message", {}) or {}).get("content", "")
            
            print(f"API response message content:\n{content}\n")
            action = (content or "").strip() 
            if not action:
                action = "O = I"
            usage = response.get("usage", {})
            stats = {
                "prompt_tokens": usage.get("prompt_tokens", 0),
                "completion_tokens": usage.get("completion_tokens", 0),
                "total_tokens": usage.get("total_tokens", 0),
            }

        # Clean up the action
        action = (action or "").strip().strip("`").strip()
        if action.startswith("python"):
            action = action[6:].strip()

        # Detect multi-line grid literal: e.g. O = [[0,1,...],\n  [2,3,...]].
        # If the response starts with an assignment to a list-of-lists,
        # collapse all lines into one so the executor can eval it.
        lines = action.split("\n")
        if len(lines) > 1 and re.match(r'^\w+\s*=\s*\[\[', lines[0]):
            # Join all lines into a single line, strip whitespace
            action = " ".join(l.strip() for l in lines)
            # Remove any trailing text after the closing brackets
            # Find the balanced end of the outer list
            depth = 0
            end_idx = None
            for ci, ch in enumerate(action):
                if ch == '[':
                    depth += 1
                elif ch == ']':
                    depth -= 1
                    if depth == 0:
                        end_idx = ci
                        break
            if end_idx is not None:
                # Keep everything up to and including the final ]
                eq_pos = action.index('=')
                action = action[:end_idx + 1]
        else:
            # Normal case: single DSL call — take first line only
            action = lines[0].strip().strip("`").strip()
            if action.startswith("python"):
                action = action[6:].strip()

        self.history.append({"role": "assistant", "content": action})
        return action, stats


print("ARCAgent class defined.")


ARCAgent class defined.


<!-- ## 8. Game Loop

The main loop mirrors the TALE-Suite game loop:
observation → agent action → environment step → repeat. -->


In [63]:
def play_puzzle(agent: ARCAgent, puzzle_id: str, data_dir: str = "data",
                max_steps: int = 30, verbose: bool = True,
                log_dir: str = "logs", num_distractor_dsls: int = 0) -> dict:
    """Run the agent on one ARC puzzle. Returns a trajectory dict."""

    # ── Setup logging ─────────────────────────────────────────────
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, f"{puzzle_id}.log")
    log_file = open(log_path, "w")

    def log(msg: str = ""):
        """Write to log file and optionally print."""
        log_file.write(msg + "\n")
        log_file.flush()
        if verbose:
            print(msg)

    puzzle = load_puzzle(puzzle_id, data_dir)
    solver_info = parse_solver(puzzle_id, SOLVERS_SOURCE_PATH)
    test_input = puzzle["test"][0]["input"]
    test_output = puzzle["test"][0]["output"]

    # ── Inject distractor DSLs to increase difficulty ─────────────
    import random
    gt_dsl_funcs = set(solver_info["dsl_funcs"])  # preserve original
    if num_distractor_dsls > 0:
        # Pick N random DSL functions strictly NOT in the ground-truth set
        candidates = [fn for fn in DSL_FUNCTIONS if fn not in gt_dsl_funcs]
        n = min(num_distractor_dsls, len(candidates))
        distractors = set(random.sample(candidates, n))
        assert distractors.isdisjoint(gt_dsl_funcs), "Distractors overlap with GT!"
        solver_info["dsl_funcs"] = gt_dsl_funcs | distractors
    else:
        distractors = set()

    # Shuffled display order: concat GT + distractors then permute
    dsl_display_order = sorted(solver_info["dsl_funcs"])
    random.shuffle(dsl_display_order)
    solver_info["dsl_funcs_display"] = dsl_display_order

    executor = DSLExecutor(test_input, solver_info)
    system_prompt = build_system_prompt()
    agent.reset(system_prompt)
    obs = build_initial_observation(puzzle, solver_info)

    trajectory = {
        "puzzle_id": puzzle_id,
        "model": agent.model,
        "max_steps": max_steps,
        "gt_num_steps": solver_info["num_steps"],
        "gt_dsl_funcs": sorted(gt_dsl_funcs),
        "distractor_dsl_funcs": sorted(distractors),
        "num_distractor_dsls": num_distractor_dsls,
        "system_prompt": system_prompt,
        "planning_response": "",
        "steps": [],
        "solved": False,
        "final_match": (0, 0),
    }

    log(f'\n{"="*60}')
    log(f'PUZZLE: {puzzle_id}  |  GT steps: {solver_info["num_steps"]}  |  '
        f'GT DSLs: {len(gt_dsl_funcs)}  |  Distractors: {len(distractors)}  |  '
        f'Total DSLs shown: {len(solver_info["dsl_funcs"])}')
    log(f'{"="*60}')

    # ── Log GT vs distractor DSL breakdown ────────────────────────
    log_file.write(f"\nGT DSL functions ({len(gt_dsl_funcs)}):\n")
    for fn in sorted(gt_dsl_funcs):
        doc = DSL_DOCS.get(fn, fn)
        log_file.write(f"  [GT] {doc}\n")
    log_file.write(f"\nDistractor DSL functions ({len(distractors)}):\n")
    for fn in sorted(distractors):
        doc = DSL_DOCS.get(fn, fn)
        log_file.write(f"  [DISTRACTOR] {doc}\n")
    log_file.write(f"\nDisplay order shown to agent:\n")
    for fn in dsl_display_order:
        tag = "GT" if fn in gt_dsl_funcs else "DISTRACTOR"
        log_file.write(f"  [{tag}] {fn}\n")
    log_file.write("\n")
    log_file.flush()

    # ── Log initial observation ────────────────────────────────────
    log_file.write("\n--- INITIAL OBSERVATION ---\n")
    log_file.write(obs + "\n")
    log_file.write("--- END INITIAL OBSERVATION ---\n\n")
    log_file.flush()

    # ── Planning round: ask agent to reason about the pattern ─────
    planning_prompt = (
        obs + "\n\n"
        "Before writing any code, please analyze the training input-output pairs "
        "and describe:\n"
        "1. What transformation pattern/rule do you see?\n"
        "2. How will you apply the available DSL functions to implement it?\n"
        "3. What is your step-by-step plan?\n\n"
        "Respond with your analysis (free-form text is fine for this round)."
    )
    agent.history.append({"role": "user", "content": planning_prompt})
    planning_response = agent._call_with_retry(agent.history)
    planning_text = ""
    if planning_response and "choices" in planning_response:
        planning_text = (planning_response.get("choices", [{}])[0]
                         .get("message", {}) or {}).get("content", "") or ""
    agent.history.append({"role": "assistant", "content": planning_text})
    trajectory["planning_response"] = planning_text

    log("\n--- PLANNING ROUND ---")
    log(planning_text)
    log("--- END PLANNING ---\n")

    # After planning, the next user message tells the agent to start coding
    obs = (
        "Good. Now begin executing your plan. "
        "Remember: output EXACTLY ONE line of DSL code per turn.\n\n"
        + executor.get_state_summary() + "\n"
        "\nAvailable DSL functions:\n"
        + "\n".join(f"  {DSL_DOCS[fn]}" for fn in solver_info.get("dsl_funcs_display", sorted(solver_info["dsl_funcs"])) if fn in DSL_DOCS)
        + "\n\nWrite the next DSL call:"
    )

    for step in range(1, max_steps + 1):
        action, stats = agent.act(obs)

        log(f'  Step {step:2d} | {action}')

        # Execute the DSL call (this is the 'environment step')
        success, exec_msg = executor.execute_line(action)
        log_file.write(f"    exec: {exec_msg}\n")

        # Check output
        solved = False
        match_cells, total_cells = 0, 0
        if executor.has_output():
            output_grid = executor.get_output()
            try:
                match_cells, total_cells = count_matching_cells(output_grid, test_output)
                solved = grids_match(output_grid, test_output)
            except Exception:
                pass

        trajectory["steps"].append({
            "step": step,
            "action": action,
            "success": success,
            "exec_msg": exec_msg,
            "has_output": executor.has_output(),
            "match_cells": match_cells,
            "total_cells": total_cells,
            "solved": solved,
            "tokens": stats.get("total_tokens", 0),
        })

        if solved:
            trajectory["solved"] = True
            trajectory["final_match"] = (match_cells, total_cells)
            log(f'  >>> SOLVED at step {step}! ({match_cells}/{total_cells} cells)')
            break

        # ── Build next observation (feedback to the agent) ──────────
        obs_parts = []
        if success:
            obs_parts.append(f"Executed: {action}")
            obs_parts.append(f"Result: {exec_msg}")
        else:
            obs_parts.append(f"ERROR executing: {action}")
            obs_parts.append(f"Error: {exec_msg}")
            obs_parts.append("Please try again with a valid DSL call.")

        obs_parts.append("")
        obs_parts.append(executor.get_state_summary())

        if executor.has_output():
            obs_parts.append(f'\nOutput accuracy: {match_cells}/{total_cells} cells match')
            obs_parts.append("The output is NOT correct yet. Keep refining.")

        obs_parts.append("\nAvailable DSL functions:")
        for fname in solver_info.get("dsl_funcs_display", sorted(solver_info["dsl_funcs"])):
            if fname in DSL_DOCS:
                obs_parts.append(f"  {DSL_DOCS[fname]}")

        obs_parts.append("\nWrite the next DSL call:")
        obs = "\n".join(obs_parts)

        # Log the feedback observation
        log_file.write(f"    --- feedback obs (step {step}) ---\n")
        log_file.write(obs + "\n")
        log_file.flush()

    if not trajectory["solved"]:
        if executor.has_output():
            try:
                mc, tc = count_matching_cells(executor.get_output(), test_output)
                trajectory["final_match"] = (mc, tc)
            except Exception:
                pass
        mc, tc = trajectory["final_match"]
        log(f'  >>> NOT solved after {max_steps} steps. Accuracy: {mc}/{tc}')

    # ── Write final summary to log ─────────────────────────────────
    log_file.write(f"\n{'='*60}\n")
    log_file.write(f"RESULT: {'SOLVED' if trajectory['solved'] else 'FAILED'}\n")
    mc, tc = trajectory['final_match']
    log_file.write(f"Final match: {mc}/{tc} cells\n")
    log_file.write(f"Steps used: {len(trajectory['steps'])}/{max_steps}\n")
    log_file.write(f"GT steps: {solver_info['num_steps']}\n")
    log_file.close()

    if verbose:
        print(f'  [log saved to {log_path}]')

    return trajectory


print("play_puzzle() defined.")


play_puzzle() defined.


<!-- ## 9. Pretty-Print Trajectory -->


In [64]:
def pretty_print_trajectory(traj: dict):
    """Print a formatted summary of an ARC puzzle trajectory."""
    print("=" * 70)
    print("ARC-AGI TRAJECTORY SUMMARY")
    print("=" * 70)
    print(f'  Puzzle ID    : {traj["puzzle_id"]}')
    print(f'  Model        : {traj["model"]}')
    print(f'  Max Steps    : {traj["max_steps"]}')
    print(f'  GT Steps     : {traj["gt_num_steps"]}')
    print(f'  GT DSL Funcs : {traj["gt_dsl_funcs"]}')
    print(f'  Solved       : {traj["solved"]}')
    mc, tc = traj["final_match"]
    print(f'  Final Match  : {mc}/{tc} cells')
    print()

    for entry in traj["steps"]:
        status = "OK" if entry["success"] else "ERR"
        solved_mark = " SOLVED!" if entry["solved"] else ""
        match_str = ""
        if entry["has_output"]:
            match_str = f' | Match: {entry["match_cells"]}/{entry["total_cells"]}'
        print(f'  Step {entry["step"]:2d} [{status}] {entry["action"]}{match_str}{solved_mark}')
    print()


print("pretty_print_trajectory() defined.")


pretty_print_trajectory() defined.


<!-- ## 10. Configuration & Run

Set your API credentials below and choose which puzzles to evaluate. -->


In [65]:
# --- Configuration (adjust these) ---
API_URL = "https://tritonai-api.ucsd.edu/v1/chat/completions"
API_KEY = os.environ.get("OPENAI_API_KEY")
print(API_KEY)         # <-- Replace with your TritonAPI key
MODEL = "api-gpt-oss-120b"
#MODEL = "moonshotai.kimi-k2.5"
MAX_STEPS = 20
SEED = 42
DATA_DIR = "data"
NUM_DISTRACTOR_DSLS = 0  # Extra irrelevant DSLs shown to agent (increase to raise difficulty)
LOG_DIR = "logs/oss_strictDSLonly"  # Directory to save logs (change if using distractors)

# --- Choose puzzles to run ---
from puzzle_ids import *
PUZZLE_IDS = ALL_IDS    

# --- Choose puzzles to run ---
# from puzzle_ids import *        
# PUZZLE_IDS = ["d406998b"]            # <--- ADD THIS LINE


# PUZZLE_IDS = TEST_IDS              # all 27 test puzzles
# PUZZLE_IDS = TEST_IDS[0:3]          # quick test: first 3
# PUZZLE_IDS = ["68b16354"]            # single puzzle for debugging


sk-CkbGlxqt3QqNPZbmWIUoTA


In [52]:
print(f"Running ARCAgent on {len(PUZZLE_IDS)} puzzles with model {MODEL}")
print("puzzle ids to run:", PUZZLE_IDS)

Running ARCAgent on 131 puzzles with model moonshotai.kimi-k2.5
puzzle ids to run: ['c9e6f938', 'a5313dff', '3906de3d', '50cb2852', 'd0f5fe59', '1f876c06', '68b16354', '0ca9ddb6', '2dc579da', 'ae4f1146', 'c59eb873', '6fa7a44f', 'a699fb00', '05f2a901', '5521c0d9', 'a416b8f3', '0b148d64', '007bbfb7', '42a50994', '7fe24cdd', 'ac0a08a4', 'a740d043', 'be94b721', '40853293', '7b6016b9', '48d8fb45', '5bd6f4ac', 'e9614598', 'c909285e', 'd631b094', '3618c87e', '445eab21', '74dd1130', '3aa6fb7a', '39a8645d', '5117e062', '4c4377d9', '496994bd', '46f33fce', '363442ee', '44d8ac46', '4347f46a', '22eb0ac0', 'c1d99e64', 'b27ca6d3', 'f25fbde4', '4258a5f9', 'd23f8c26', '3af2c5a8', 'b94a9452', '543a7ed5', '1e0a9b12', '44f52bb0', 'b60334d2', '2dee498d', '67385a82', '8efcae92', '1cf80156', 'ea32f347', '28bf18c6', '90c28cc7', 'd2abd087', 'e98196ab', 'dae9d2b5', '6f8cd79b', 'f76d97a5', 'bb43febb', 'b6afb2da', '46442a0e', 'b1948b0a', '32597951', '662c240a', '7b7f7511', '5168d44c', 'c8f0f002', '8d5021e8', '6d0

<!-- ### Run the Agent -->


In [66]:
agent = ARCAgent(
    api_url=API_URL,
    api_key=API_KEY,
    model=MODEL,
    seed=SEED,
    temperature=0.5,
    max_tokens=131072,
)

results = []
skipped = []
for pid in PUZZLE_IDS:
    # ── Firewall: skip puzzles with large test input grids ─────
    puzzle_data = load_puzzle(pid, DATA_DIR)
    test_grid = puzzle_data["test"][0]["input"]
    h, w = len(test_grid), len(test_grid[0]) if test_grid else 0
    if h > 10 or w > 10:
        print(f'\nSKIPPING {pid}: test input grid too large ({h}x{w} > 10x10)')
        skipped.append((pid, h, w))
        continue

    print(f'\n{"#"*60}')
    print(f'# Playing puzzle: {pid}')
    print(f'{"#"*60}')
    traj = play_puzzle(agent, pid, data_dir=DATA_DIR, max_steps=MAX_STEPS, verbose=True,
                       num_distractor_dsls=NUM_DISTRACTOR_DSLS, log_dir=LOG_DIR)
    results.append(traj)

# ── Summary ──
print("\n" + "=" * 60)
print("OVERALL RESULTS")
print("=" * 60)
if skipped:
    print(f'Skipped (grid > 10x10): {len(skipped)}')
    for pid, h, w in skipped:
        print(f'  {pid}: {h}x{w}')
solved_count = sum(1 for r in results if r["solved"])
print(f'Solved: {solved_count} / {len(results)}')
for r in results:
    mc, tc = r["final_match"]
    status = "SOLVED" if r["solved"] else "FAILED"
    steps_used = len(r["steps"])
    print(f'  {r["puzzle_id"]}: {status} | Steps: {steps_used}/{r["max_steps"]} | '
          f'GT: {r["gt_num_steps"]} | Match: {mc}/{tc}')



############################################################
# Playing puzzle: c9e6f938
############################################################

PUZZLE: c9e6f938  |  GT steps: 2  |  GT DSLs: 2  |  Distractors: 0  |  Total DSLs shown: 2

--- PLANNING ROUND ---
1. **Transformation pattern**  
   Each output grid is twice as wide as the input (3 × 3 → 3 × 6). The left half of the output is exactly the original grid, and the right half is the original grid reflected horizontally (mirrored along the vertical axis). In other words:  
   `output = concatenate_horizontally(input, vertical_mirror(input))`.

2. **Applying the DSL**  
   - Use `vmirror(I)` to obtain the vertically‑mirrored version of the input grid.  
   - Use `hconcat(I, <mirrored>)` to place the original grid and its mirror side‑by‑side.

3. **Step‑by‑step plan**  
   - Compute the mirrored grid and store it (e.g., `x1`).  
   - Concatenate the original grid `I` with `x1` to produce the final output `O`.

The next DSL cal

<!-- ### Inspect Trajectory -->


In [18]:
if results:
    pretty_print_trajectory(results[-1])


ARC-AGI TRAJECTORY SUMMARY
  Puzzle ID    : dae9d2b5
  Model        : api-gpt-oss-120b
  Max Steps    : 20
  GT Steps     : 6
  GT DSL Funcs : ['combine', 'fill', 'lefthalf', 'ofcolor', 'righthalf']
  Solved       : True
  Final Match  : 9/9 cells

  Step  1 [OK] O = fill(lefthalf(I), SIX, combine(ofcolor(lefthalf(I), FOUR), ofcolor(righthalf(I), THREE))) | Match: 9/9 SOLVED!



In [67]:
total_puzzles = len(results)
solved_count = 0
total_match_rate = 0.0

print(f"{'Puzzle ID':<12} | {'Solved':<8} | {'Match':<12} | {'Rate':<7}")
print("-" * 45)

for traj in results:
    pid = traj['puzzle_id']
    solved = traj['solved']
    mc, tc = traj['final_match']
    
    if solved:
        solved_count += 1
        
    # Calculate the match rate for this specific puzzle
    rate = (mc / tc) if tc > 0 else 0.0
    total_match_rate += rate
    
    # Print the row
    print(f"{pid:<12} | {str(solved):<8} | {f'{mc}/{tc}':<12} | {rate:6.2%}")

print("=" * 45)
print("SUMMARY:")
if total_puzzles > 0:
    print(f"Solved: {solved_count}/{total_puzzles} ({(solved_count/total_puzzles):.1%})")
    avg_rate = total_match_rate / total_puzzles
    print(f"Average Matching Cell Rate: {avg_rate:.2%}")
else:
    print("No results to summarize.")


Puzzle ID    | Solved   | Match        | Rate   
---------------------------------------------
c9e6f938     | True     | 18/18        | 100.00%
a5313dff     | False    | 67/81        | 82.72%
3906de3d     | False    | 84/100       | 84.00%
1f876c06     | True     | 100/100      | 100.00%
68b16354     | True     | 49/49        | 100.00%
0ca9ddb6     | True     | 81/81        | 100.00%
ae4f1146     | True     | 9/9          | 100.00%
c59eb873     | True     | 100/100      | 100.00%
6fa7a44f     | True     | 18/18        | 100.00%
a699fb00     | True     | 100/100      | 100.00%
a416b8f3     | True     | 40/40        | 100.00%
007bbfb7     | True     | 81/81        | 100.00%
7fe24cdd     | True     | 36/36        | 100.00%
ac0a08a4     | True     | 144/144      | 100.00%
a740d043     | True     | 4/4          | 100.00%
be94b721     | True     | 12/12        | 100.00%
48d8fb45     | True     | 9/9          | 100.00%
5bd6f4ac     | True     | 9/9          | 100.00%
d631b094     | True     |

<!-- ## 11. Run All Test Puzzles (Optional)

Uncomment the cell below to evaluate on all 27 test puzzles. -->


In [ ]:
# from puzzle_ids import TEST_IDS

# agent_full = ARCAgent(
#     api_url=API_URL,
#     api_key=API_KEY,
#     model=MODEL,
#     seed=SEED,
# )

# all_results = []
# for pid in TEST_IDS:
#     traj = play_puzzle(agent_full, pid, data_dir=DATA_DIR, max_steps=MAX_STEPS)
#     all_results.append(traj)

# solved = sum(1 for r in all_results if r["solved"])
# print(f'Solved: {solved} / {len(all_results)}')
# for r in all_results:
#     mc, tc = r["final_match"]
#     s = "SOLVED" if r["solved"] else "FAILED"
#     print(f'  {r["puzzle_id"]}: {s} | Match: {mc}/{tc} | Steps: {len(r["steps"])}/{r["gt_num_steps"]}')
